# Satellite Building Change Detection (LEVIR-CD)

CS57600 Machine Learning, Fall 2026 · Raviteja Gundu, Joel Adu-Kwarteng

Predict pixel-level building-change masks from pairs of satellite images, comparing an image-difference baseline, an early-fusion U-Net (U-Net-EF) and a Siamese U-Net.

**How to run**
1. Upload this notebook to Colab and choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells. Allow Drive access when prompted.

Data, checkpoints and results are stored on Google Drive under `CONFIG["project_root"]`, so they survive between Colab sessions. Each task adds one section to this notebook.

## Task 1 · Setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib
import numpy as np
import PIL
import torch

# All settings live here. Later tasks add their own entries (data, baseline, models).
CONFIG = {
    "project_root": "/content/drive/MyDrive/levir-change-detection",  # persistent storage
    "local_root": "/content/local",  # fast VM disk, wiped when the session ends
    "seed": 42,
    "deterministic": False,  # True makes cuDNN deterministic (slower)
}

In [ ]:
def set_seed(seed: int, deterministic: bool = False) -> None:
    """Seed Python, NumPy and torch (CPU and all GPUs)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


def seed_worker(worker_id: int) -> None:
    """DataLoader worker_init_fn: derive NumPy/Python seeds from torch's per-worker seed."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass(frozen=True)
class Paths:
    """Standard locations under the Drive project root, plus a fast local scratch folder."""

    root: Path
    local: Path

    @property
    def raw(self) -> Path:  # LEVIR-CD as downloaded: train/val/test/{A,B,label}
        return self.root / "data" / "raw"

    @property
    def tiles_zip(self) -> Path:  # 256x256 tiles (Task 3)
        return self.root / "data" / "tiles.zip"

    @property
    def stats_file(self) -> Path:  # training-set channel mean/std (Task 4)
        return self.root / "data" / "stats.json"

    @property
    def runs(self) -> Path:  # checkpoints and logs, one folder per run
        return self.root / "runs"

    @property
    def results(self) -> Path:  # metrics JSON and CSVs
        return self.root / "results"

    @property
    def figures(self) -> Path:  # figures for the report
        return self.results / "figures"

    @property
    def local_tiles(self) -> Path:  # tiles unzipped to local disk each session
        return self.local / "tiles"

    def run_dir(self, name: str) -> Path:
        return self.runs / name

    def make_dirs(self) -> "Paths":
        for p in (self.raw, self.runs, self.results, self.figures, self.local):
            p.mkdir(parents=True, exist_ok=True)
        return self

In [ ]:
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted. Run the drive.mount cell above first.")

set_seed(CONFIG["seed"], CONFIG["deterministic"])
DEVICE = get_device()
PATHS = Paths(Path(CONFIG["project_root"]), Path(CONFIG["local_root"])).make_dirs()

print(f"torch {torch.__version__} | numpy {np.__version__} | pillow {PIL.__version__} | matplotlib {matplotlib.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", DEVICE)
print()
for name in ("root", "raw", "tiles_zip", "stats_file", "runs", "results", "figures", "local_tiles"):
    print(f"{name:>11}: {getattr(PATHS, name)}")

## Task 2 · Metrics

All metrics are for the **change** class (label 1) and come from confusion counts:

| Metric | Formula |
|---|---|
| Precision | TP / (TP + FP) |
| Recall | TP / (TP + FN) |
| F1 (= Dice) | 2·TP / (2·TP + FP + FN) |
| IoU | TP / (TP + FP + FN) |
| Accuracy | (TP + TN) / all pixels |

Dataset-level results add up the counts over every test pixel first and compute the metrics once (not an average of per-image scores).

Zero denominators: if a mask has no change and the prediction has none either (TP = FP = FN = 0), precision, recall, F1 and IoU are 1.0. Any other zero denominator gives 0.0. This mostly matters for per-image scores on no-change tiles.

Inputs must already be binary (0/1 or bool). Threshold probabilities first. Tensors are counted on their own device, so validation on the GPU never copies masks to the CPU.

In [ ]:
METRIC_NAMES = ("precision", "recall", "f1", "iou", "accuracy")


def _squeeze_channel(x):
    """(N, 1, H, W) -> (N, H, W), so model outputs and masks line up."""
    return x[:, 0] if x.ndim == 4 and x.shape[1] == 1 else x


def _as_binary(x, name):
    """Return a bool tensor/array. Raises if values other than 0/1 are present."""
    if torch.is_tensor(x):
        if x.dtype != torch.bool:
            if not ((x == 0) | (x == 1)).all():
                raise ValueError(f"{name} must be binary (0/1). Threshold probabilities first.")
            x = x != 0
        return x
    x = np.asarray(x)
    if x.dtype != bool:
        if not np.isin(x, (0, 1)).all():
            raise ValueError(f"{name} must be binary (0/1). Threshold probabilities first.")
        x = x != 0
    return x


def confusion_counts(pred, target):
    """Return (tp, fp, fn, tn) as Python ints for binary masks of the same shape.

    Accepts numpy arrays or torch tensors. If either is a tensor, counting happens on
    that tensor's device.
    """
    if torch.is_tensor(pred) or torch.is_tensor(target):
        device = pred.device if torch.is_tensor(pred) else target.device
        pred = torch.as_tensor(pred, device=device)
        target = torch.as_tensor(target, device=device)
    pred, target = _squeeze_channel(pred), _squeeze_channel(target)
    if tuple(pred.shape) != tuple(target.shape):
        raise ValueError(f"Shape mismatch: pred {tuple(pred.shape)} vs target {tuple(target.shape)}")
    p, t = _as_binary(pred, "pred"), _as_binary(target, "target")
    tp = int((p & t).sum())
    fp = int((p & ~t).sum())
    fn = int((~p & t).sum())
    tn = int(t.numel() if torch.is_tensor(t) else t.size) - tp - fp - fn
    return tp, fp, fn, tn


def metrics_from_counts(tp, fp, fn, tn):
    """Precision, recall, F1, IoU and accuracy from confusion counts, plus the counts."""
    empty = tp + fp + fn == 0  # nothing to find and nothing predicted

    def ratio(num, den):
        if den == 0:
            return 1.0 if empty else 0.0
        return num / den

    total = tp + fp + fn + tn
    return {
        "precision": ratio(tp, tp + fp),
        "recall": ratio(tp, tp + fn),
        "f1": ratio(2 * tp, 2 * tp + fp + fn),
        "iou": ratio(tp, tp + fp + fn),
        "accuracy": (tp + tn) / total if total else 1.0,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


class ConfusionAccumulator:
    """Running TP/FP/FN/TN over many batches. Call compute() for dataset-level metrics."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.tp = self.fp = self.fn = self.tn = 0

    def update(self, pred, target):
        tp, fp, fn, tn = confusion_counts(pred, target)
        self.tp += tp
        self.fp += fp
        self.fn += fn
        self.tn += tn
        return self

    def compute(self):
        return metrics_from_counts(self.tp, self.fp, self.fn, self.tn)

    def __repr__(self):
        return f"ConfusionAccumulator(tp={self.tp}, fp={self.fp}, fn={self.fn}, tn={self.tn})"


def per_image_metrics(pred, target, names=None):
    """Metrics for each image in a batch shaped (N, H, W) or (N, 1, H, W).

    Returns a list of dicts. If names is given, each dict also gets a "name" key.
    """
    pred, target = _squeeze_channel(pred), _squeeze_channel(target)
    if len(pred) != len(target):
        raise ValueError(f"Batch size mismatch: {len(pred)} vs {len(target)}")
    rows = []
    for i in range(len(pred)):
        row = metrics_from_counts(*confusion_counts(pred[i], target[i]))
        if names is not None:
            row = {"name": names[i], **row}
        rows.append(row)
    return rows


def format_metrics(m):
    return "  ".join(f"{k}={m[k]:.4f}" for k in METRIC_NAMES)

### Demo: hand-computed toy example

A 4×4 target with 4 changed pixels and a prediction with 4 changed pixels, 3 of them correct:
TP = 3, FP = 1, FN = 1, TN = 11, so precision = recall = F1 = 0.75, IoU = 3/5 = 0.6 and accuracy = 14/16 = 0.875.

The cell checks these values and a few other cases, and raises an error if anything is off.

In [ ]:
target = np.array([[0, 0, 0, 0],
                   [0, 1, 1, 0],
                   [0, 1, 1, 0],
                   [0, 0, 0, 0]])
pred = np.array([[0, 0, 0, 0],
                 [0, 1, 1, 1],
                 [0, 1, 0, 0],
                 [0, 0, 0, 0]])


def check(label, got, expected):
    ok = all(np.isclose(got[k], v) for k, v in expected.items())
    print(f"{'OK  ' if ok else 'FAIL'} {label:<38} {format_metrics(got)}")
    if not ok:
        raise AssertionError(f"{label}: expected {expected}, got {got}")


# 1. Hand-computed example
check("toy 4x4 (numpy)", metrics_from_counts(*confusion_counts(pred, target)),
      {"tp": 3, "fp": 1, "fn": 1, "tn": 11,
       "precision": 0.75, "recall": 0.75, "f1": 0.75, "iou": 0.6, "accuracy": 0.875})

# 2. Same example as torch tensors on the GPU, shaped like model output (N, 1, H, W)
t_pred = torch.tensor(pred, device=DEVICE)[None, None]
t_target = torch.tensor(target, device=DEVICE)[None]
check(f"toy 4x4 (torch on {DEVICE.type})", metrics_from_counts(*confusion_counts(t_pred, t_target)),
      {"precision": 0.75, "recall": 0.75, "f1": 0.75, "iou": 0.6})

# 3. Edge cases
zeros, ones = np.zeros((4, 4), int), np.ones((4, 4), int)
check("no change, nothing predicted", metrics_from_counts(*confusion_counts(zeros, zeros)),
      {"precision": 1, "recall": 1, "f1": 1, "iou": 1, "accuracy": 1})
check("all change, nothing predicted", metrics_from_counts(*confusion_counts(zeros, ones)),
      {"precision": 0, "recall": 0, "f1": 0, "iou": 0, "accuracy": 0})
check("no change, everything predicted", metrics_from_counts(*confusion_counts(ones, zeros)),
      {"precision": 0, "recall": 0, "f1": 0, "iou": 0, "accuracy": 0})
check("perfect prediction", metrics_from_counts(*confusion_counts(target, target)),
      {"precision": 1, "recall": 1, "f1": 1, "iou": 1, "accuracy": 1})

# 4. Accumulating random batches equals computing on everything at once
rng = np.random.default_rng(0)
big_pred = rng.integers(0, 2, size=(10, 32, 32))
big_target = rng.integers(0, 2, size=(10, 32, 32))
acc = ConfusionAccumulator()
for start in range(0, 10, 3):  # uneven batches: 3, 3, 3, 1
    acc.update(big_pred[start:start + 3], big_target[start:start + 3])
check("batched == all at once", acc.compute(), metrics_from_counts(*confusion_counts(big_pred, big_target)))

# 5. Probabilities are rejected instead of silently miscounted
try:
    confusion_counts(np.array([[0.2, 0.9]]), np.array([[0, 1]]))
    raise AssertionError("non-binary input was accepted")
except ValueError:
    print("OK   non-binary input rejected")

# Per-image metrics for a small batch
print()
for row in per_image_metrics(np.stack([pred, zeros]), np.stack([target, zeros]), names=["toy", "empty"]):
    print(f"{row['name']:>6}: {format_metrics(row)}")